In [1]:
# 2조 API 키 - 제일 먼저 실행해주기
import os 
os.environ["OPENAI_API_KEY"] = ""

# 실행 시 필요한 필수 자료 
- "./datasets/menu_description_qa.json"

# Chroma 벡터스토어 구축 (한 번 실행 후 재실행 X)

In [2]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
import json

# 기존 크로마 db 삭제
import shutil
import os

# if os.path.exists("./datasets/chroma_db"):
#     shutil.rmtree("./datasets/chroma_db")  # 💣 기존 데이터 완전 삭제


def build_vectorstore(json_path, persist_directory):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    docs = [
        Document(
            page_content=item["description"],
            metadata={
                "menu_path": item["menu_path"],
                "table_info": item.get("table_info", ""),
                "graph_info": item.get("graph_info", ""),
                "map_info": item.get("map_info", ""),
                "url": item["url"],
                "login_required": item["login_required"],
                "source" : item["source"]
            }
        )

        for item in data
    ]

    splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=300)
    split_docs = splitter.split_documents(docs)

    vectordb = Chroma.from_documents(
        documents=split_docs,
        embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
        persist_directory=persist_directory
    )

if __name__ == "__main__":
    # build_vectorstore("./datasets/enriched_menu_description_qa.json", "./datasets/chroma_db")
    build_vectorstore("./datasets/menu_description_qa.json", "./datasets/chroma_db_1500_300")


# 전처리 코드 (아래부터는 실행할 필요 X)

## 메뉴 경로 설명 파일

In [16]:
import re
import pandas as pd

def split_menu_blocks(text_path):
    with open(text_path, "r", encoding="utf-8") as f:
        text = f.read()

    # 메뉴 경로 기준으로 블록 분리
    blocks = re.findall(r"(‘.+?메뉴는.*?)(?=\n‘|\Z)", text, re.DOTALL)

    # DataFrame 생성
    df = pd.DataFrame(blocks, columns=["menu_description_block"])
    df.to_csv('./datasets/menu_description.csv',index=False)
    return df


In [17]:
menu_description = split_menu_blocks('./datasets/Menu_description.txt')
menu_description.head(1)

,menu_description_block
0,"‘국내통계 → 한국무역 → 수출입 총괄 → 총괄’ 메뉴는 연도별, 분기별, 월별 단..."


In [18]:
menu_description['menu_description_block'][38]

'‘해외무역통계 → 아시아 → 대만 → 국가별’ 메뉴는 특정 연도 기준으로 대만과 각 국가 간의 수출입 실적을 비교할 수 있는 메뉴이다. 국가별로 수출금액, 수입금액, 수출입 증감률, 무역수지가 제공되며, 정렬 기준에 따라 주요 교역국과의 무역 흐름을 한눈에 파악할 수 있다. \n'

In [19]:
menu_description["menu_path"] = menu_description["menu_description_block"].str.extract(r"(‘.+?’)")[0].str.strip("‘’")
menu_description.head(1)

,menu_description_block,menu_path
0,"‘국내통계 → 한국무역 → 수출입 총괄 → 총괄’ 메뉴는 연도별, 분기별, 월별 단...",국내통계 → 한국무역 → 수출입 총괄 → 총괄


In [20]:
# 컬럼 위치 재정렬 
menu_description = menu_description[["menu_path", "menu_description_block"]]
menu_description.head(1)

,menu_path,menu_description_block
0,국내통계 → 한국무역 → 수출입 총괄 → 총괄,"‘국내통계 → 한국무역 → 수출입 총괄 → 총괄’ 메뉴는 연도별, 분기별, 월별 단..."


## QA 파일

In [21]:
with open('./datasets/Menu_QA.txt', "r", encoding="utf-8") as f:
    text = f.read()

# \n\n 또는 \r\n\r\n 기준으로 블록 나누기
blocks = [block.strip() for block in text.strip().split("\n\n") if block.strip()]

# DataFrame 생성
menu_qa = pd.DataFrame(blocks, columns=["menu_qa"])

menu_qa.head(1)
menu_qa['menu_qa'][0]
# menu_qa

'질문: 한국 전체 수출입 규모나 종합적인 무역 현황 통계를 보고 싶어. 국가별, 품목별 구분 없이 전체 수출입 데이터가 필요해.\n답변: 아래 메뉴에서 해당 데이터를 확인할 수 있습니다:\n▶ 국내통계 → 한국무역 → 수출입 총괄 → 총괄\n🔗 https://stat.kita.net/stat/kts/sum/SumImpExpTotalList.screen'

In [22]:
# menu_qa["menu_path"] = menu_qa["menu_qa"].str.extract(r"▶ (.+?)\n").strip()
# menu_qa["menu_path"] = menu_qa["menu_qa"].str.extract(r"▶ (.+?)\n")[0].strip()
menu_qa["menu_path"] = menu_qa["menu_qa"].str.extract(r"▶ (.+?)\n")[0].str.strip()
menu_qa = menu_qa[["menu_path", "menu_qa"]]
menu_qa.head(1)

,menu_path,menu_qa
0,국내통계 → 한국무역 → 수출입 총괄 → 총괄,"질문: 한국 전체 수출입 규모나 종합적인 무역 현황 통계를 보고 싶어. 국가별, 품..."


## 경로 기준 데이터 프레임 합치기(3)

In [23]:
# 1단계: 중간에 있는 → - → 제거
menu_qa["menu_path"] = menu_qa["menu_path"].str.replace(r"→\s*-\s*→", "→", regex=True)

# 2단계: 마지막에 남은 → - 제거
menu_qa["menu_path"] = menu_qa["menu_path"].str.replace(r"→\s*-\s*$", "", regex=True)


## 병합 기준 컬럼 문자, 공백 전처리 
- 공백 제거
- 메뉴에서는 -> 메뉴는 통일 (txt 파일)
- 그래프/표/지도는 -> 그래프/표/지도 구성은 통일 (txt파일)
- 자사통계 → 수출입 실적 (국가별/ 총괄/ 품목별) 수정

In [24]:
# 양쪽 공백 제거 + 특수문자 제거 + 표준화
for df in [menu_description, menu_qa]:
    df["menu_path"] = df["menu_path"].str.strip()                           # 앞뒤 공백 제거
    df["menu_path"] = df["menu_path"].str.replace(r"\s+", " ", regex=True) # 이중 공백 제거
    df["menu_path"] = df["menu_path"].str.replace("\u200b", "")             # zero-width space 제거


In [25]:
menu_description_qa = pd.merge(menu_description, menu_qa, on="menu_path", how="outer")
# df.head()
menu_description_qa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   menu_path               184 non-null    object
 1   menu_description_block  184 non-null    object
 2   menu_qa                 184 non-null    object
dtypes: object(3)
memory usage: 4.4+ KB


In [26]:
missing_rows = menu_description_qa[menu_description_qa["menu_description_block"].isna() | menu_description_qa["menu_qa"].isna()]
# missing_rows = df[df["menu_description_block"].isna()]
missing_rows

,menu_path,menu_description_block,menu_qa


## 저장

In [28]:
menu_description_qa.to_csv('./datasets/menu_description_qa.csv',index=False)

# 메뉴 설명과 FAQ 추출 + JSON 저장 

In [5]:
import pandas as pd
import re
import json

def parse_csv_to_json_with_login_levels(csv_path, json_output):
    df = pd.read_csv(csv_path)
    items = []

    for _, row in df.iterrows():
        menu_path = row.get("menu_path", "")
        
        # desc_block = row.get("menu_description_block", "")
        desc_block_full = row.get("menu_description_block", "")
        # 구성 키워드가 등장하기 전까지만 추출
        desc_block = re.split(r"(지도 구성은|표 구성은|그래프 구성은)", desc_block_full)[0].strip()

        qa_block = row.get("menu_qa", "")

        # URL 추출
        url_match = re.search(r"🔗\s*(https?://[^\s\n]+)", str(qa_block))
        url = url_match.group(1) if url_match else None

        # 로그인 등급 분기 처리
        login_text = str(qa_block)
        if "기업회원(유료회원사)" in login_text:
            login_required = "유료회원"
        elif "일반 회원 로그인이 필요합니다" in login_text:
            login_required = "일반회원"
        else:
            login_required = "비회원가능"

        # 메뉴 path 분리
        path_parts = menu_path.split(" → ")
        parts = path_parts + [""] * (4 - len(path_parts))  # ensure 4 parts
        main, sub, category, menu = parts[:4]

        # 표/그래프 분리
        table_info = ""
        graph_info = ""
        # if "표 구성은" in desc_block:
        #     parts = desc_block.split("표 구성은")
        #     desc_block = parts[0].strip()
        #     table_info = parts[1].split("그래프 구성은")[0].strip() if "그래프 구성은" in parts[1] else parts[1].strip()
        # if "그래프 구성은" in desc_block:
        #     parts = desc_block.split("그래프 구성은")
        #     desc_block = parts[0].strip()
        #     graph_info = parts[1].strip()

        table_match = re.search(r"(표 구성은.+?)(지도 구성은|그래프 구성은|$)", desc_block_full, re.DOTALL)
        if table_match:
            table_info = table_match.group(1).strip()
     
        graph_match = re.search(r"(그래프 구성은.+?)(지도 구성은|표 구성은|$)", desc_block_full, re.DOTALL)
        if graph_match:
            graph_info = graph_match.group(1).strip()

        map_match = re.search(r"(지도 구성은.+?)(그래프 구성은|표 구성은|$)", desc_block_full, re.DOTALL)
        if map_match:
            map_info = map_match.group(1).strip()

        
        # FAQ 구성
        faq = []
        if table_info:
            faq.append({"question": "표에 대해 설명해줘", "answer": table_info})
        if graph_info:
            faq.append({"question": "그래프에 대해 설명해줘", "answer": graph_info})
        if map_info:
            faq.append({"question": "지도에 대해 설명해줘", "answer": map_info})
        if isinstance(qa_block, str) and qa_block.strip():
            qa_pairs = re.findall(r"질문:\s*(.*?)\n답변:\s*(.*?)(?=\n질문:|\Z)", qa_block, re.DOTALL)
            for question, answer in qa_pairs:
                faq.append({"question": question.strip(), "answer": answer.strip()})

        description_enriched = desc_block.strip()
        if faq:
            description_enriched += "\n\n[자주 묻는 질문]\n"
            for qa in faq:
                description_enriched += f"Q. {qa['question']}\nA. {qa['answer']}\n"

        items.append({
            "main_menu": main,
            "sub_menu": sub,
            "category": category,
            "menu": menu,
            "menu_path": menu_path,
            "description_original": desc_block.strip(),
            "description": description_enriched,
            "table_info": table_info,        
            "graph_info": graph_info,        
            "map_info": map_info,        
            "url": url,
            "login_required": login_required,
            "source": "경로",
            # "faq": faq
        })

    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False, indent=2)

# 실행
parse_csv_to_json_with_login_levels(
    "./datasets/menu_description_qa.csv",
    "./datasets/menu_description_qa.json"
)
